# TD 1 · Représenter : du modèle linéaire au réseau de neurones

**Bloc 3 : deep learning · Séance 8**

---

## Objectifs

À la fin de ce TD, vous serez capable de :

1. montrer, chiffres à l'appui, pourquoi aucun modèle linéaire ne résout XOR ;
2. vérifier qu'un neurone n'est rien d'autre qu'une logistic regression ;
3. coder des activation functions et leurs dérivées, et relier la saturation au
   vanishing gradient ;
4. démontrer numériquement que des couches linéaires empilées restent linéaires ;
5. calculer un forward pass à la main en suivant les formes des tableaux ;
6. construire un réseau qui approche une fonction, et dire ce que l'universal
   approximation theorem ne garantit pas.

## Prérequis

- `00c_NumPy_Essentials` : produit matriciel `@`, formes, `axis` ;
- séance 3 (fabriquer des features), séance 4 (gradient descent),
  séance 5 (sigmoïde, cross-entropy) ;
- cours : partie 1 du diaporama (séance 8).

## Concepts mobilisés

XOR · neurone (poids, biais, activation) · sigmoïde, tanh, ReLU, Leaky ReLU ·
saturation et vanishing gradient · effondrement linéaire · forward pass ·
universal approximation.

## Déroulé

| Exercice | Contenu | Durée | Type | Niveau |
|---|---|---|---|---|
| 1.1 | XOR et logistic regression | 10 min | guidé | ★☆☆ |
| 1.2 | Réparer XOR à la main | 8 min | modifier | ★☆☆ |
| 1.3 | Un neurone est une logistic regression | 10 min | compléter | ★☆☆ |
| 1.4 | Activations et dérivées | 12 min | compléter, expérimenter | ★★☆ |
| 1.5 | Empiler des couches linéaires | 12 min | expérimenter, comparer | ★★☆ |
| 1.6 | Le forward pass à la main | 10 min | calculer, vérifier | ★★☆ |
| 1.7 | Universal approximation, par construction | 13 min | résoudre, analyser | ★★☆ à ★★★ |
| | Synthèse et questions | 5 min | | |

## Résultats attendus

- Sur XOR, la logistic regression plafonne à une accuracy de 0,50 ; la meilleure
  droite possible atteint 0,75 ; avec la feature x₁·x₂, on atteint 1,00.
- Votre neurone codé à la main reproduit `predict_proba` de scikit-learn.
- La dérivée de la sigmoïde ne dépasse jamais 0,25 : sur dix couches, le gradient
  est multiplié par moins d'un millionième.
- Deux couches linéaires sans activation donnent exactement une seule couche.
- Forward pass de la fiche : z₁ = [0,5 ; 0,6], a₂ ≈ 0,525, loss ≈ 0,644.
- L'erreur d'approximation diminue quand on ajoute des unités, et l'approximation
  s'effondre hors de la zone vue.

## Mode d'emploi

- Les cellules marquées `# TODO` sont à compléter. Une cellule de vérification suit
  presque toujours : ne passez pas à la suite tant qu'elle n'affiche pas « OK ».
- Chaque exercice suit la même boucle : **explication courte → exemple → code →
  votre exercice → analyse → question**.
- Dans le texte, les nombres s'écrivent avec une virgule décimale ; Python les
  affiche avec un point.

## Mise en route

In [ ]:
import sys
from math import comb
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from src.data import SEED, set_seed

set_seed(SEED)
# Des décimales lisibles plutôt que la notation scientifique : tout le TD consiste
# à comparer des tableaux affichés avec des calculs faits sur papier.
np.set_printoptions(precision=6, suppress=True)
# Sur quatre points, MLPClassifier signale parfois une convergence incomplète. Ce
# message n'apporte rien ici ; on le masque pour garder les sorties lisibles.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("prêt, seed", SEED)

---

## Exercice 1.1 · XOR et logistic regression

*Guidé · 10 min · ★☆☆*

**Explication.** XOR (« ou exclusif ») : quatre points, deux features, et une
étiquette qui vaut 1 quand **exactement une** des deux entrées vaut 1. Un modèle
linéaire sépare le plan par une seule droite : d'un côté il prédit 1, de l'autre 0.

| x₁ | x₂ | y |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**Exemple.** On ajuste une logistic regression, exactement comme en séance 5.

In [ ]:
# XOR : les deux points de classe 1 sont sur une diagonale, les deux points de
# classe 0 sur l'autre.
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

lineaire = LogisticRegression().fit(X_xor, y_xor)
print("Logistic regression sur XOR")
print(f"  prédictions  : {lineaire.predict(X_xor)}")
print(f"  vérité       : {y_xor}")
print(f"  accuracy     : {lineaire.score(X_xor, y_xor):.2f}")
# Les quatre probabilités à 0,5 : le modèle n'a aucune opinion.
print(f"  probabilités : {lineaire.predict_proba(X_xor)[:, 1].round(4)}")

![XOR : logistic regression contre réseau](../assets/generated/cours_08_10_fr/f01_xor.png)

À gauche, ce que la logistic regression a appris : 0,5 partout. À droite, ce
qu'un petit réseau apprendra au TD 2 : deux bandes, donc une frontière qui n'est
pas une droite.

**À vous.** Avant d'accuser l'algorithme, testez la famille de modèles elle-même :
une droite peut-elle mieux faire ?

In [ ]:
# TODO : écrivez exactitude_droite(w1, w2, b), qui renvoie la proportion de points
# de XOR bien classés par la règle « classe 1 si w1*x1 + w2*x2 + b > 0 ».
# Essayez ensuite au moins trois droites de votre choix.


def exactitude_droite(w1, w2, b):
    # TODO
    raise NotImplementedError


# for w1, w2, b in [(1, 1, -0.5), (____, ____, ____), (____, ____, ____)]:
#     print(w1, w2, b, exactitude_droite(w1, w2, b))

In [ ]:
# Recherche exhaustive sur une grille de 41 x 41 x 41 droites (près de 69 000).
# Si une droite pouvait tout séparer, on la trouverait ici.
grille = np.linspace(-2, 2, 41)
meilleure = max(exactitude_droite(a, b, c)
                for a in grille for b in grille for c in grille)
print(f"meilleure accuracy d'une droite sur la grille : {meilleure:.2f}")

**Analyse.** Aucune droite ne dépasse **3 points sur 4**. Ce n'est pas un défaut
d'optimisation : la famille des modèles linéaires ne contient tout simplement pas
de solution.

Remarquez pourtant que la logistic regression affiche 0,50, et non 0,75. Elle ne
cherche pas la meilleure accuracy : elle minimise la cross-entropy (séance 5).

**Question 1.1.** Une droite peut classer 3 points sur 4. Pourquoi la logistic
regression préfère-t-elle prédire 0,5 partout ?

---

## Exercice 1.2 · Réparer XOR à la main

*Modifier · 8 min · ★☆☆*

**Explication.** En séance 3, vous avez fabriqué des features. Il manque ici une
combinaison **non linéaire** des entrées : le produit x₁·x₂, qui ne vaut 1 que pour
le point (1, 1).

**À vous.** Ajoutez cette colonne et réajustez le modèle.

In [ ]:
# TODO : construisez X_plus = [x1, x2, x1*x2] avec np.column_stack, ajustez
# LogisticRegression(C=100) et affichez l'accuracy, les probabilités et les
# coefficients. (C=100 signifie une pénalité faible : voir l'analyse.)
X_plus = None
avec_produit = None

In [ ]:
# Le coût de la méthode manuelle : le nombre de produits à envisager explose.
for d in (2, 10, 50):
    print(f"  {d:3d} features : {comb(d, 2):6d} produits de deux, "
          f"{comb(d, 3):7d} produits de trois")

**Analyse.** Avec la bonne feature, un modèle linéaire résout XOR : les deux
premiers coefficients sont positifs et égaux, le troisième fortement négatif. Le
modèle a retrouvé la règle « x₁ + x₂ - 2·x₁x₂ », à un facteur d'échelle près.

Mais il a fallu **savoir** quelle combinaison ajouter. Avec 50 features, il y a
1 225 produits de deux et 19 600 produits de trois : on ne peut pas les essayer à
la main. Un réseau de neurones est une machine à **apprendre** ces combinaisons.

**Question 1.2.** Pourquoi faut-il `C=100` ici, alors que `C=1` suffisait souvent
en séance 5 ?

---

## Exercice 1.3 · Un neurone est une logistic regression

*Compléter · 10 min · ★☆☆*

**Explication.** Un neurone calcule une somme pondérée de ses entrées, ajoute un
biais, puis applique une activation function g :

$$ a = g(\mathbf{w}^\top \mathbf{x} + b) $$

Avec g = sigmoïde, c'est **exactement** la prédiction d'une logistic regression.
Le réseau n'ajoutera qu'une chose : empiler ces unités, avec une non-linéarité
entre les couches.

![Un neurone : entrées, poids, biais, activation](../assets/diagrams/s08_networks/nn_feedforward.png)

*Schéma de la bibliothèque du cours (provenance non établie, usage en classe).*

In [ ]:
def sigmoide(z):
    # Écrase n'importe quel réel dans (0, 1) : la sortie se lit comme une probabilité.
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
# TODO : écrivez neurone(X, w, b, g) = g(X @ w + b). Puis, avec g = sigmoide et les
# coefficients de avec_produit (coef_[0] et intercept_[0]), comparez vos
# probabilités à avec_produit.predict_proba(X_plus)[:, 1].


def neurone(X, w, b, g):
    # TODO
    raise NotImplementedError


p_neurone = None
p_sklearn = None

In [ ]:
# Vérification : les deux calculs doivent coïncider à la précision machine.
ecart = np.abs(np.asarray(p_neurone) - np.asarray(p_sklearn)).max()
print(f"  écart maximal : {ecart:.1e}   {'OK' if ecart < 1e-12 else 'à revoir'}")

**Analyse.** L'écart est nul (ou de l'ordre de 10⁻¹⁶) : `predict_proba` n'est rien
d'autre que cette ligne. Un neurone isolé ne sait donc pas plus de choses qu'une
logistic regression : sur les données brutes de XOR, il plafonnerait lui aussi.

**Question 1.3.** Qu'est-ce qu'un réseau ajoute à ce neurone, et pourquoi cela
suffit-il à se passer de la feature x₁·x₂ fabriquée à la main ?

---

## Exercice 1.4 · Activations et dérivées

*Compléter, expérimenter · 12 min · ★★☆*

**Explication.** Quatre activations courantes :

| nom | g(z) | g'(z) |
|---|---|---|
| sigmoïde | 1 / (1 + e⁻ᶻ) | g(z)·(1 - g(z)) |
| tanh | tanh z | 1 - tanh² z |
| ReLU (*rectified linear unit*, unité linéaire rectifiée) | max(0, z) | 1 si z > 0, sinon 0 |
| Leaky ReLU (ReLU « qui fuit ») | z si z > 0, sinon α·z | 1 si z > 0, sinon α |

Pourquoi s'intéresser aux **dérivées** ? Parce que la backpropagation (TD 2)
multiplie le gradient par g'(z) à chaque couche qu'il traverse.

In [ ]:
# TODO : complétez les sept fonctions. np.tanh, np.maximum et np.where suffisent.


def d_sigmoide(z):
    raise NotImplementedError


def tanh(z):
    raise NotImplementedError


def d_tanh(z):
    raise NotImplementedError


def relu(z):
    raise NotImplementedError


def d_relu(z):
    raise NotImplementedError


def leaky_relu(z, alpha=0.1):
    raise NotImplementedError


def d_leaky_relu(z, alpha=0.1):
    raise NotImplementedError

In [ ]:
# Vérification par finite differences (voir TD 2) : on compare chaque dérivée codée
# à la pente mesurée numériquement. On évite z proche de 0, où ReLU n'est pas
# dérivable.
z_test = np.random.default_rng(0).uniform(-4, 4, 200)
z_test = z_test[np.abs(z_test) > 1e-3]
h = 1e-6
for nom, g, dg in [("sigmoïde", sigmoide, d_sigmoide), ("tanh", tanh, d_tanh),
                   ("ReLU", relu, d_relu), ("Leaky ReLU", leaky_relu, d_leaky_relu)]:
    try:
        pente = (g(z_test + h) - g(z_test - h)) / (2 * h)
        ecart = np.abs(pente - dg(z_test)).max()
        print(f"  {nom:10s} écart max {ecart:.1e}   {'OK' if ecart < 1e-6 else 'à revoir'}")
    except NotImplementedError:
        print(f"  {nom:10s} à compléter")

In [ ]:
z = np.linspace(-5, 5, 401)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nom, g, dg in [("sigmoïde", sigmoide, d_sigmoide), ("tanh", tanh, d_tanh),
                   ("ReLU", relu, d_relu), ("Leaky ReLU", leaky_relu, d_leaky_relu)]:
    axes[0].plot(z, g(z), lw=2, label=nom)
    axes[1].plot(z, dg(z), lw=2, label=nom)
axes[0].set(title="g(z)", xlabel="z", ylim=(-1.3, 3))
axes[1].set(title="g'(z) : le facteur appliqué au gradient", xlabel="z")
axes[1].axhline(0.25, color="grey", ls=":")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**À vous : expérimenter.** Un réseau profond multiplie ces dérivées couche après
couche. Mesurez ce que cela donne.

In [ ]:
# TODO :
#  (a) valeur maximale de d_sigmoide sur np.linspace(-6, 6, 1201) ;
#  (b) ce maximum élevé à la puissance 10 (dix couches sigmoïdes, dans le meilleur cas) ;
#  (c) la même chose si chaque unité est à z = 2 (déjà un peu saturée) ;
#  (d) la même chose pour ReLU avec des unités actives (z = 2).

**Analyse.** La dérivée de la sigmoïde ne dépasse jamais **0,25**. Après dix
couches, le gradient est multiplié au mieux par 0,25¹⁰ ≈ 10⁻⁶, et bien moins dès
que les unités saturent : les premières couches n'apprennent presque plus. C'est le
**vanishing gradient**. Avec ReLU, une unité active laisse passer le gradient intact
(facteur 1) : c'est ce qui a rendu les réseaux profonds entraînables.

Le revers : une unité ReLU dont l'entrée est négative pour **tous** les exemples
ne reçoit plus aucun gradient et ne peut plus revenir (dead unit, phénomène de
dying ReLU, TD 2). Leaky ReLU garde un petit facteur α pour l'éviter.

**Question 1.4.** Pourquoi garde-t-on une sigmoïde en **sortie** d'un classifieur
binaire alors qu'on l'évite dans les hidden layers ?

---

## Exercice 1.5 · Empiler des couches linéaires

*Expérimenter, comparer · 12 min · ★★☆*

**Explication.** Sans activation, deux couches calculent

$$ W_2(W_1\mathbf{x} + b_1) + b_2 = (W_2W_1)\mathbf{x} + (W_2b_1 + b_2) $$

c'est à dire **une seule** couche linéaire, de poids W₂W₁. Vérifions-le sur des
nombres, puis mesurons ce que cela coûte sur XOR.

In [ ]:
# TODO :
#  1. avec rng = np.random.default_rng(1), tirez W1 (4, 2), b1 (4,), W2 (1, 4), b2 (1,) ;
#  2. tirez 100 entrées X_alea de forme (100, 2) ;
#  3. calculez deux_couches = (X_alea @ W1.T + b1) @ W2.T + b2 (aucune activation) ;
#  4. calculez W = W2 @ W1, b = W2 @ b1 + b2, puis une_couche = X_alea @ W.T + b ;
#  5. affichez l'écart maximal entre deux_couches et une_couche.

**À vous : comparer.** Entraînons de vrais réseaux sur XOR, avec
`MLPClassifier` de scikit-learn utilisé ici comme une boîte noire (au TD 2 vous
écrirez vous-même ce qu'il fait).

In [ ]:
# TODO : pour 5 seeds (random_state=0..4), entraînez sur XOR :
#   A. MLPClassifier(hidden_layer_sizes=(8,) * 10, activation="identity", solver="lbfgs", max_iter=2000)
#   B. MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs", max_iter=2000)
#   C. MLPClassifier(hidden_layer_sizes=(4,), activation="relu", solver="lbfgs", max_iter=2000)
# et affichez les 5 valeurs d'accuracy de chacun.

**Analyse.** Le réseau A a 10 couches et plusieurs centaines de paramètres, mais il
ne dépasse jamais **0,75**, l'accuracy de la meilleure droite trouvée en 1.1 :
c'est une droite, calculée de façon coûteuse. Le réseau B, avec une seule couche de
quatre unités tanh, résout XOR pour toutes les seeds.

Le réseau C mérite un regard : avec ReLU, le résultat **dépend de la seed**. Sur
quatre points, une unité ReLU peut s'éteindre définitivement au début de
l'entraînement. Vous mesurerez ce phénomène au TD 2 (exercice 2.6).

**Question 1.5.** Quel est le minimum qu'un réseau doit contenir pour battre la
logistic regression sur XOR ? Que dire d'un réseau profond sans activation ?

---

## Exercice 1.6 · Le forward pass à la main

*Calculer, vérifier · 10 min · ★★☆*

**Explication.** Le réseau de la fiche de la séance 8 : deux entrées, deux unités
cachées ReLU, une sortie sigmoïde, neuf paramètres. Un seul exemple,
**x** = [1, 0], de classe y = 1.

![Le réseau de la fiche](../assets/generated/cours_08_10_fr/f04_reseau_fiche.png)

Le forward pass enchaîne quatre calculs : z₁ = W₁x + b₁, a₁ = ReLU(z₁),
z₂ = W₂a₁ + b₂, a₂ = σ(z₂), puis la loss L = -log(a₂) puisque y = 1.

**Méthode.** La plupart des erreurs de ce bloc sont des erreurs de **forme** (un
nombre qui rencontre le mauvais poids). Donnez d'abord les formes, ensuite les
nombres.

In [ ]:
x = np.array([1.0, 0.0])
y_vrai = 1.0
W1 = np.array([[0.5, -0.5],
               [0.5, 0.5]])
b1 = np.array([0.0, 0.1])
W2 = np.array([[1.0, -1.0]])
b2 = np.array([0.2])

In [ ]:
# TODO (1) : sans rien calculer, donnez la forme (un tuple) de chaque quantité.
formes = {"W1": None, "b1": None, "z1": None, "a1": None,
          "W2": None, "b2": None, "z2": None, "a2": None}

# TODO (2) : faites le calcul sur papier, puis recopiez vos valeurs (listes ou nombres).
z1_main, a1_main, z2_main, a2_main, perte_main = None, None, None, None, None

In [ ]:
# Vérification : le calcul de référence, fait par NumPy.
z1 = W1 @ x + b1
a1 = np.maximum(0.0, z1)
z2 = W2 @ a1 + b2
a2 = sigmoide(z2)
perte = float(-np.log(a2[0]))
reference = {"W1": W1, "b1": b1, "z1": z1, "a1": a1, "W2": W2, "b2": b2, "z2": z2, "a2": a2}


def verifier(nom, valeur, attendu, tol=2e-3):
    if valeur is None:
        print(f"  {nom:6s} : à compléter")
    elif np.allclose(np.ravel(valeur), np.ravel(attendu), atol=tol):
        print(f"  {nom:6s} : OK")
    else:
        print(f"  {nom:6s} : à revoir (vous : {valeur}, attendu : {np.round(attendu, 6)})")


print("Formes")
for nom, forme in formes.items():
    etat = "à compléter" if forme is None else (
        "OK" if tuple(forme) == reference[nom].shape else f"à revoir ({reference[nom].shape})")
    print(f"  {nom:6s} : {etat}")
print("Valeurs")
for nom, valeur, attendu in [("z1", z1_main, z1), ("a1", a1_main, a1), ("z2", z2_main, z2),
                             ("a2", a2_main, a2), ("loss", perte_main, perte)]:
    verifier(nom, valeur, attendu)
print(f"\nréférence : a2 = {a2[0]:.6f}, loss = {perte:.6f}")

**Analyse.** Le réseau attribue une probabilité d'environ 0,525 à la classe 1 alors
que la vraie classe est 1 : il est presque sans opinion, et sa loss (0,644) est
proche de log 2. Tout l'enjeu du TD 2 est de savoir **dans quel sens** bouger
chacun des neuf paramètres pour faire baisser cette loss.

**Question 1.6.** Sans calcul, et en regardant seulement la figure : faut-il
augmenter ou diminuer b₂ pour faire baisser la loss ? Et le poids -1,0 qui relie h₂
à la sortie ?

---

## Exercice 1.7 · Universal approximation, par construction

*Résoudre, analyser · 13 min · ★★☆ (mini-défi ★★★)*

**Explication.** Une unité ReLU décalée, relu(x - t), est une **charnière** : nulle
avant t, puis une droite. Une somme pondérée de charnières est donc une fonction
linéaire par morceaux, et avec assez de morceaux on épouse n'importe quelle courbe
continue sur un intervalle. C'est l'intuition de l'**universal approximation
theorem** : un réseau à une seule hidden layer, avec une activation non
polynomiale, peut approcher aussi près qu'on veut toute fonction continue sur un
domaine borné (Cybenko, 1989 ; Hornik, 1991 ; Leshno et al., 1993 pour ReLU).

**Exemple.** La valeur absolue s'écrit exactement avec deux unités :
|x| = relu(x) + relu(-x).

In [ ]:
xs = np.linspace(-3, 3, 7)
print("  x                   :", xs)
print("  relu(x) + relu(-x)  :", np.maximum(0, xs) + np.maximum(0, -xs))
print("  |x|                 :", np.abs(xs))

**À vous.** Construisez (sans entraînement) un réseau à une hidden layer qui
approche sin(x) sur [0, 2π].

In [ ]:
# TODO : complétez reseau_charnieres.


def reseau_charnieres(x, noeuds, cible):
    """Approche `cible` par une somme de ReLU, une unité par point de rupture.

    On veut la fonction linéaire par morceaux qui passe par (t, cible(t)) pour
    chaque noeud t.
      pente du morceau i       : p_i = (cible(t_{i+1}) - cible(t_i)) / (t_{i+1} - t_i)
      coefficient de l'unité i : c_0 = p_0, puis c_i = p_i - p_{i-1}
      sortie                   : cible(t_0) + somme_i c_i * relu(x - t_i)
    (la somme porte sur tous les noeuds sauf le dernier)
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Erreur maximale sur [0, 2π] quand on double le nombre d'unités.
x_fin = np.linspace(0, 2 * np.pi, 4001)
for k in (3, 6, 12, 24, 48):
    noeuds = np.linspace(0, 2 * np.pi, k + 1)
    erreur = np.abs(reseau_charnieres(x_fin, noeuds, np.sin) - np.sin(x_fin)).max()
    print(f"  {k:3d} unités : erreur maximale {erreur:.4f}")

# Hors de la zone : on évalue le réseau à 12 unités sur [2π, 3π].
noeuds12 = np.linspace(0, 2 * np.pi, 13)
x_hors = np.linspace(2 * np.pi, 3 * np.pi, 500)
erreur_hors = np.abs(reseau_charnieres(x_hors, noeuds12, np.sin) - np.sin(x_hors)).max()
print(f"\n  12 unités, hors de la zone [2π, 3π] : erreur maximale {erreur_hors:.2f}")

![Approximation par charnières](../assets/generated/cours_08_10_fr/f03a_approximation.png)

![Et hors de la zone](../assets/generated/cours_08_10_fr/f03b_extrapolation.png)

**Analyse.** Chaque fois qu'on double le nombre d'unités, l'erreur est divisée par
environ **quatre** : l'approximation s'améliore sans limite. Mais hors de la zone
vue, le réseau prolonge sa dernière droite et l'erreur explose : l'approximation
ne vaut que là où il y avait des points.

**Mini-défi (★★★).** Avec le même budget de **8 unités**, placez les points de
rupture vous-même pour battre l'erreur de l'espacement régulier. Indice :
où la courbe se plie-t-elle le plus ?

In [ ]:
# TODO : déplacez les noeuds intérieurs (gardez 0 et 2π aux extrémités, 9 noeuds).
noeuds_perso = np.linspace(0, 2 * np.pi, 9)
erreur_reguliere = np.abs(reseau_charnieres(x_fin, np.linspace(0, 2 * np.pi, 9), np.sin)
                          - np.sin(x_fin)).max()
erreur_perso = np.abs(reseau_charnieres(x_fin, noeuds_perso, np.sin) - np.sin(x_fin)).max()
print(f"  espacement régulier : {erreur_reguliere:.4f}   vos noeuds : {erreur_perso:.4f}")

**Question 1.7.** Le théorème garantit qu'un réseau à une hidden layer **peut**
approcher toute fonction continue. Citez quatre choses qu'il ne garantit **pas**.

---

## Synthèse

- Sur XOR, **aucune droite** ne dépasse 3 points sur 4 ; la logistic regression,
  qui minimise la cross-entropy, s'arrête à 0,5 partout.
- Fabriquer x₁·x₂ résout XOR, mais la méthode manuelle ne passe pas à l'échelle : un
  réseau **apprend** les combinaisons utiles.
- Un neurone sigmoïde **est** une logistic regression.
- La dérivée de la sigmoïde ne dépasse pas 0,25 : empilée, elle **éteint** le
  gradient. ReLU laisse passer un facteur 1, au risque de dead units.
- Sans activation, **dix couches valent une** : la non-linéarité n'est pas
  optionnelle.
- Une somme de charnières ReLU approche n'importe quelle courbe sur un intervalle,
  et rien au-delà : **représenter n'est pas apprendre, ni généraliser**.

## Questions de compréhension

1. Pourquoi « 0,50 » sur XOR n'est-il pas un échec d'optimisation ?
2. Qu'est-ce qu'un réseau ajoute à une logistic regression ?
3. Montrez qu'un réseau à deux couches sans activation est une couche unique.
4. Pourquoi ReLU a-t-elle remplacé la sigmoïde dans les hidden layers ?
5. Que garantit l'universal approximation theorem, et que ne garantit-il pas ?

## Pour aller plus loin

- Goodfellow, Bengio et Courville (2016), *Deep Learning*, MIT Press, chapitre 6.
  Accès libre en ligne : https://www.deeplearningbook.org/
- Nielsen, M. (2015), *Neural Networks and Deep Learning*, chapitre 4, « A visual
  proof that neural nets can compute any function ». Licence CC BY-NC 3.0.
  http://neuralnetworksanddeeplearning.com/chap4.html
- Cybenko, G. (1989), *Approximation by superpositions of a sigmoidal function*,
  Mathematics of Control, Signals and Systems 2(4), 303-314.
- Hornik, K. (1991), *Approximation capabilities of multilayer feedforward
  networks*, Neural Networks 4(2), 251-257.
- Leshno, M., Lin, V., Pinkus, A. et Schocken, S. (1993), *Multilayer feedforward
  networks with a nonpolynomial activation function can approximate any function*,
  Neural Networks 6(6), 861-867.

**Prochain TD :** faire apprendre ce réseau, à la main puis en PyTorch.